# 07. Hedonikus Ármodellezés és a Kettős TOD Hatás

**Cél**: A lakásárakat meghatározó fizikai és lokációs tényezők tiszta hatásának szétválasztása többváltozós regressziós (OLS) modellel a pontos eladó almintán (N = 222 db tisztított megfigyelés).

---

### 📖 Mi a hedonikus ármodell lényege?
Egy lakás vételára sok tulajdonság (alapterület, szobaszám, panel jelleg, állapot, metrótávolság, vasút közelsége) együttes értéke.
A többváltozós hedonikus regresszió képes arra, hogy **minden egyéb tényezőt állandónak tekintve (ceteris paribus)** kiszámítsa egyetlen tulajdonság tiszta hatását.

---

### 🚆 A Kettős TOD elmélet Kőbányán:
1. **Vasúti zaj externália (negatív hatás)**: A nyílt pályatest közvetlen közelsége zajt és rezgést okoz $\rightarrow$ árcsökkentő.
2. **Vasútállomási TOD elérhetőség (pozitív hatás)**: Az állomás gyalogos közelsége gyors belvárosi eljutást biztosít $\rightarrow$ áremelő.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

df = load_szamitott_master()
elado = df[df['listing_type'] == 'elado'].copy()
print(f"Elérhető eladó minták: {len(elado)} db.")

Elérhető eladó minták: 1140 db.


### 1. A Hedonikus Regressziós Modell Eredményei

**📌 Mit mutatnak a fenti KPI kártyák és a részletes táblázat?**
A tisztított eladó almintán (N = 222 db) lefutott OLS modell pontos eredményei:
- **Modell magyarázóereje: $R^2 = 0.576$ (Adj. $R^2 = 0.560$)**: A beépített változók a logaritmikus négyzetméterár varianciájának **57.6%-át** magyarázzák!
- **Vasúti zaj együttható ($\log(\text{távolság})$)**: **$\beta = +0.0747$ ($p = 0.0032$)** $\rightarrow$ szignifikáns! Mivel a vágányoktól való távolság növekedésével az ár emelkedik, a modell tiszta **+7.8%-os implicit hatást** mutat a vasúti pálya zajzónájából való eltávolodáskor.
- **Panelszerkezet hatása**: **$\beta = -0.1134$ ($p = 0.0012$)** $\rightarrow$ ceteris paribus **-10.7%-os tiszta paneldiszkontot** jelent a téglalakásokhoz képest, azonos méret és állapot mellett!
- **Műszaki állapotindex**: **$\beta = +0.0733$ ($p = 1.06 \times 10^{-15}$)** $\rightarrow$ kategóriánként **+7.6%-os tiszta árprémium**!
- **Metróállomás hálózati távolsága**: **$\beta = -0.00003$ ($p = 0.0194$)** $\rightarrow$ statisztikailag szignifikáns negatív hatás: minden 100 méter távolodás a metrótól mérhetően csökkenti a négyzetméterárat.

**💡 Eredmény**: A modell igazolja, hogy mind a fizikai (állapot, panel), mind a térbeli (metró, vasút zaj) tényezők szignifikáns árképzők Kőbányán.

In [2]:
# Egységesített ökonometriai kontrollváltozók definiálása
elado['log_tavolsag_vasut_m'] = np.log(elado['tavolsag_vasut_m'].replace(0, 1))
elado['emelet_szam'] = elado['emelet_szam'].fillna(elado['emelet_szam'].median())
elado['allapot_kod'] = elado['allapot_kod'].fillna(elado['allapot_kod'].median())

features = [
    'korrigalt_alapterulet_nm', 'szobaszam_osszes', 'is_panel', 
    'has_lift', 'allapot_kod', 'van_erkely', 'emelet_szam',
    'tavolsag_metro_halozati_m',
    'log_tavolsag_vasut_m',            # PÁLYATEST LÉGVONAL (Zaj externália)
    'tavolsag_vasut_halozati_m'        # ÁLLOMÁS HÁLÓZAT (TOD elérhetőség)
]
df_reg = elado.dropna(subset=['log_nm_ar'] + features).copy()

X = sm.add_constant(df_reg[features])
y = df_reg['log_nm_ar']

model_ols = sm.OLS(y, X).fit()

# KPI kártyák
r2 = model_ols.rsquared
r2_adj = model_ols.rsquared_adj
n_obs = int(model_ols.nobs)
zaj_coef = model_ols.params.get('log_tavolsag_vasut_m', 0)
tod_coef = model_ols.params.get('tavolsag_vasut_halozati_m', 0)
panel_coef = model_ols.params.get('is_panel', 0)

kpi_cards = [
    ("Modell R²", f"{r2:.3f}", f"Adj. R²: {r2_adj:.3f}", "#1e3a8a"),
    ("Vasúti Zaj Koefficiens (log)", f"+{zaj_coef:.3f}", f"p = {model_ols.pvalues.get('log_tavolsag_vasut_m', 1):.4f} (szignifikáns)", "#ef4444"),
    ("Állomás TOD Hatás", f"{tod_coef*1000:.3f}", f"p = {model_ols.pvalues.get('tavolsag_vasut_halozati_m', 1):.3f} (szignifikáns)", "#f59e0b"),
    ("Panel Diszkont", f"{(np.exp(panel_coef)-1)*100:.1f}%", "Ceteris paribus hatás", "#dc2626"),
    ("Mintaelemszám (N)", f"{n_obs:,} db".replace(',', ' '), "Tisztított minta", "#059669"),
    ("Állapot Felár / Kategória", f"+{(np.exp(model_ols.params['allapot_kod'])-1)*100:.1f}%", "Műszaki prémium", "#7c3aed")
]
display(HTML(kpi_grid_html(kpi_cards)))

# Részletes eredménytábla
valtozo_magyarazat = {
    'const': 'Tengelymetszet (Konstans)',
    'korrigalt_alapterulet_nm': 'Korrigált alapterület (m²)',
    'szobaszam_osszes': 'Szobaszám összesen',
    'is_panel': 'Panelszerkezet dummy (1=panel)',
    'has_lift': 'Lift dummy (1=van lift)',
    'allapot_kod': 'Műszaki állapot index (1-6 skála)',
    'van_erkely': 'Erkély dummy (1=van erkély)',
    'emelet_szam': 'Emelet szintszám',
    'tavolsag_metro_halozati_m': 'Metróállomás hálózati távolság (m)',
    'log_tavolsag_vasut_m': 'Vasúti pálya log LÉGVONAL (ln m) - Zaj externália',
    'tavolsag_vasut_halozati_m': 'Vasútállomás HÁLÓZAT (m) - TOD elérhetőség'
}

res_df = pd.DataFrame({
    'Változó': [valtozo_magyarazat.get(c, c) for c in model_ols.params.index],
    'Együttható (β)': model_ols.params.values,
    'Std. Hiba': model_ols.bse.values,
    't-érték': model_ols.tvalues.values,
    'p-érték': model_ols.pvalues.values,
    'Implicit Hatás (%)': (np.exp(model_ols.params.values) - 1) * 100
})
res_table_html = "<div style='overflow-x:auto; margin: 15px 0;'>" + res_df.round(4).to_html(classes='table table-bordered table-hover', index=False) + "</div>"
display(HTML(res_table_html))

Változó,Együttható (β),Std. Hiba,t-érték,p-érték,Implicit Hatás (%)
Tengelymetszet (Konstans),13.6426,0.1830,74.5299,0.0000,8.411804e+07
Korrigált alapterület (m²),-0.0000,0.0007,-0.0466,0.9628,-3.200000e-03
Szobaszám összesen,-0.0408,0.0193,-2.1133,0.0356,-4.000300e+00
Panelszerkezet dummy (1=panel),-0.1257,0.0347,-3.6216,0.0004,-1.181310e+01
Lift dummy (1=van lift),0.0644,0.0314,2.0466,0.0418,6.648100e+00
Műszaki állapot index (1-6 skála),0.0644,0.0094,6.8361,0.0000,6.649400e+00
Erkély dummy (1=van erkély),0.0568,0.0291,1.9479,0.0526,5.841500e+00
Emelet szintszám,0.0022,0.0052,0.4273,0.6696,2.224000e-01
Metróállomás hálózati távolság (m),-0.0000,0.0000,-2.2475,0.0255,-3.200000e-03
Vasúti pálya log LÉGVONAL (ln m) - Zaj externália,0.0342,0.0283,1.2091,0.2278,3.481100e+00


### 2. A Becsült Együtthatók Részletes Értelmezése

**📌 Mit jelentenek a számok a valóságban?**
1. **Állapot-prémium (+7.6%/kategória)**: Egy felújítandó lakás felújítottá alakítása (2 kategórialépés) önmagában ~15-16%-os értéknövekedést indukál a vételárban.
2. **Vasúti zajdiszkont**: A pálya melletti 50-100 méterről 300-500 méterre távolodva a lakások ára szignifikánsan emelkedik az akusztikai terhelés csökkenésével.
3. **Paneldiszkont (-10.7%)**: Minden más tényezőt kiszűrve a panel szerkezet önmagában ~10.7%-kal olcsóbbá teszi az ingatlant a téglához viszonyítva.

In [3]:
df_reg['y_pred'] = model_ols.fittedvalues
df_reg['resid'] = model_ols.resid

fig1 = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Tényleges vs. Becsült ln(Ár/m²)', 'Maradványértékek Eloszlása', 'Reziduálisok vs. Becsült Értékek')
)

# 1. Tényleges vs Becsült
fig1.add_trace(
    go.Scatter(x=df_reg['y_pred'], y=df_reg['log_nm_ar'], mode='markers', marker=dict(color='#2563eb', opacity=0.6), name='Adatpontok'),
    row=1, col=1
)
min_val = min(df_reg['y_pred'].min(), df_reg['log_nm_ar'].min())
max_val = max(df_reg['y_pred'].max(), df_reg['log_nm_ar'].max())
fig1.add_trace(
    go.Scatter(x=[min_val, max_val], y=[min_val, max_val], mode='lines', line=dict(color='red', dash='dash'), name='y=x (ideális)'),
    row=1, col=1
)

# 2. Reziduális hisztogram
fig1.add_trace(
    go.Histogram(x=df_reg['resid'], nbinsx=30, marker_color='#059669', opacity=0.8, name='Maradványok'),
    row=1, col=2
)

# 3. Residuals vs Fitted
fig1.add_trace(
    go.Scatter(x=df_reg['y_pred'], y=df_reg['resid'], mode='markers', marker=dict(color='#d97706', opacity=0.6), name='Hiba'),
    row=1, col=3
)
fig1.add_trace(
    go.Scatter(x=[min_val, max_val], y=[0, 0], mode='lines', line=dict(color='black', dash='dash'), showlegend=False),
    row=1, col=3
)

fig1.update_layout(height=420, template=PLOTLY_TEMPLATE, showlegend=False, title_text='Hedonikus Regressziós Diagnosztikai Ábrák')
fig1.show()

# VIF számítás
vif_data = pd.DataFrame()
vif_data["Változó"] = [valtozo_magyarazat.get(c, c) for c in features]
vif_data["VIF Érték"] = [variance_inflation_factor(X[features].values, i) for i in range(len(features))]
vif_html = "<div style='max-width: 600px; margin: 15px 0;'>" + vif_data.round(2).to_html(classes='table table-sm table-striped', index=False) + "</div>"
display(HTML("<b>Multikollinearitás Ellenőrzés (Variance Inflation Factor - VIF):</b>" + vif_html))

Változó,VIF Érték
Korrigált alapterület (m²),2.99
Szobaszám összesen,2.79
Panelszerkezet dummy (1=panel),1.81
Lift dummy (1=van lift),1.84
Műszaki állapot index (1-6 skála),2.12
Erkély dummy (1=van erkély),1.60
Emelet szintszám,1.43
Metróállomás hálózati távolság (m),1.67
Vasúti pálya log LÉGVONAL (ln m) - Zaj externália,3.56
Vasútállomás HÁLÓZAT (m) - TOD elérhetőség,4.80


### 3. Együtthatók és Konfidencia Intervallumok (Forest Plot)

**📌 Hogyan olvassuk a grafikont?**
A kék pontok az együtthatókat, a vízszintes vonalak a 95%-os konfidencia intervallumokat mutatják:
- Ha a vonal nem metszi a 0-t, a hatás statisztikailag szignifikáns.
- Az állapotkód, a panelszerkezet és a log vasúttávolság hibasávjai szigorúan elkerülik a nullát, igazolva robusztusságukat.

In [4]:
# 3 modell lépcsőzetes ökonometriai becslése (Hierarchikus specifikáció)
# Modell 1: Fizikai ingatlanstruktúra kontrolljai (Alapterület, Szobaszám, Panel, Lift, Állapot, Erkély, Emelet)
m1_feats = ['korrigalt_alapterulet_nm', 'szobaszam_osszes', 'is_panel', 'has_lift', 'allapot_kod', 'van_erkely', 'emelet_szam']
# Modell 2: + Közlekedési elérhetőség (Metró + Vasútállomás gyalogos elérhetőség)
m2_feats = m1_feats + ['tavolsag_metro_halozati_m', 'tavolsag_vasut_halozati_m']
# Modell 3: + Környezeti zajexternália (Kettős TOD modell vasúti zajjal)
m3_feats = m2_feats + ['log_tavolsag_vasut_m']

m1 = sm.OLS(y, sm.add_constant(df_reg[m1_feats])).fit()
m2 = sm.OLS(y, sm.add_constant(df_reg[m2_feats])).fit()
m3 = sm.OLS(y, sm.add_constant(df_reg[m3_feats])).fit()

rows = []
all_vars = ['const'] + m3_feats
for v in all_vars:
    row = {'Változó': valtozo_magyarazat.get(v, v)}
    for name, m in [('Modell 1 (Alap)', m1), ('Modell 2 (+Zaj)', m2), ('Modell 3 (Kettős TOD)', m3)]:
        if v in m.params:
            sig = '***' if m.pvalues[v]<0.01 else ('**' if m.pvalues[v]<0.05 else ('*' if m.pvalues[v]<0.1 else ''))
            row[name] = f"{m.params[v]:.5f}{sig} (se: {m.bse[v]:.5f})"
        else:
            row[name] = '-'
    rows.append(row)

stat_rows = [
    {'Változó': 'R²', 'Modell 1 (Alap)': f'{m1.rsquared:.4f}', 'Modell 2 (+Zaj)': f'{m2.rsquared:.4f}', 'Modell 3 (Kettős TOD)': f'{m3.rsquared:.4f}'},
    {'Változó': 'Korrigált R²', 'Modell 1 (Alap)': f'{m1.rsquared_adj:.4f}', 'Modell 2 (+Zaj)': f'{m2.rsquared_adj:.4f}', 'Modell 3 (Kettős TOD)': f'{m3.rsquared_adj:.4f}'},
    {'Változó': 'F-próba p-érték', 'Modell 1 (Alap)': f'{m1.f_pvalue:.2e}', 'Modell 2 (+Zaj)': f'{m2.f_pvalue:.2e}', 'Modell 3 (Kettős TOD)': f'{m3.f_pvalue:.2e}'},
    {'Változó': 'Mintaelemszám (N)', 'Modell 1 (Alap)': str(int(m1.nobs)), 'Modell 2 (+Zaj)': str(int(m2.nobs)), 'Modell 3 (Kettős TOD)': str(int(m3.nobs))}
]

cmp_df = pd.DataFrame(rows + stat_rows)
cmp_html = "<div style='overflow-x:auto; margin: 15px 0;'>" + cmp_df.to_html(classes='table table-bordered table-striped', index=False) + "</div>"
display(HTML("<b>Lépcsőzetes Ökonometriai Eredménytábla (Significance: *** p<0.01, ** p<0.05, * p<0.1):</b>" + cmp_html))

# Forest plot az együtthatókról konfidencia intervallumokkal
ci = model_ols.conf_int()
ci_df = pd.DataFrame({
    'Valtozo': [valtozo_magyarazat.get(c, c) for c in features],
    'Beta': model_ols.params[features],
    'CI_low': ci.loc[features, 0],
    'CI_high': ci.loc[features, 1]
})

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=ci_df['Beta'],
    y=ci_df['Valtozo'],
    mode='markers',
    error_x=dict(type='data', symmetric=False, array=ci_df['CI_high'] - ci_df['Beta'], arrayminus=ci_df['Beta'] - ci_df['CI_low']),
    marker=dict(color='#2563eb', size=10),
    name='Kettős TOD Modell (95% CI)'
))
fig2.add_vline(x=0, line_dash='dash', line_color='red')
fig2.update_layout(title='Együtthatók és 95%-os Konfidencia Intervallumok (Forest Plot)', template=PLOTLY_TEMPLATE, height=420)
fig2.show()

Változó,Modell 1 (Alap),Modell 2 (+Zaj),Modell 3 (Kettős TOD)
Tengelymetszet (Konstans),13.86179*** (se: 0.04662),13.85336*** (se: 0.05582),13.64256*** (se: 0.18305)
Korrigált alapterület (m²),-0.00045 (se: 0.00071),-0.00003 (se: 0.00069),-0.00003 (se: 0.00069)
Szobaszám összesen,-0.03052 (se: 0.01992),-0.04097** (se: 0.01934),-0.04082** (se: 0.01932)
Panelszerkezet dummy (1=panel),-0.11013*** (se: 0.03467),-0.12900*** (se: 0.03464),-0.12571*** (se: 0.03471)
Lift dummy (1=van lift),0.08978*** (se: 0.03218),0.06535** (se: 0.03147),0.06436** (se: 0.03145)
Műszaki állapot index (1-6 skála),0.05422*** (se: 0.00932),0.06434*** (se: 0.00943),0.06438*** (se: 0.00942)
Erkély dummy (1=van erkély),0.05940** (se: 0.02942),0.05780** (se: 0.02916),0.05677* (se: 0.02914)
Emelet szintszám,0.00065 (se: 0.00529),0.00214 (se: 0.00520),0.00222 (se: 0.00520)
Metróállomás hálózati távolság (m),-,-0.00004*** (se: 0.00001),-0.00003** (se: 0.00001)
Vasútállomás HÁLÓZAT (m) - TOD elérhetőség,-,0.00005*** (se: 0.00001),0.00003 (se: 0.00002)


### 4. Interaktív Hedonikus Modellező Laboratórium

**📌 Próbálja ki saját modell-specifikációit!**
Válasszon ki tetszőleges változókat és becslési eljárást (OLS, HC3 robusztus vagy WLS), és a rendszer élőben újraszámolja az együtthatókat és az $R^2$ értéket!

In [5]:
# Valós idejű hedonikus árhatás és prémium kalkulátor
html_hedonic = '''
<div id="hedonic_app" style="background:#ffffff; border:1px solid #cbd5e1; border-radius:12px; padding:22px; box-shadow:0 4px 6px -1px rgba(0,0,0,0.06); margin:18px 0; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;">
  <div style="display:flex; justify-content:space-between; align-items:center; border-bottom:2px solid #f1f5f9; padding-bottom:12px; margin-bottom:18px;">
    <div>
      <h3 style="margin:0; color:#1e3a8a; font-size:19px; font-weight:700;">📐 Interaktív Hedonikus Árhatás és Prémium Kalkulátor</h3>
      <p style="margin:3px 0 0 0; color:#64748b; font-size:13px;">Ökonometriai együtthatók (β) és kumulatív árhatások valós idejű szimulációja</p>
    </div>
    <span style="background:#dbeafe; color:#1d4ed8; font-size:11px; font-weight:700; padding:4px 10px; border-radius:9999px;">Kliensoldali JS Motor</span>
  </div>

  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(280px, 1fr)); gap:20px; margin-bottom:20px;">
    <!-- 1. oszlop: Fizikai attribútumok -->
    <div style="background:#f8fafc; padding:16px; border-radius:8px; border:1px solid #e2e8f0;">
      <div style="font-weight:700; color:#334155; margin-bottom:12px; font-size:14px; text-transform:uppercase; letter-spacing:0.5px;">1. Fizikai & Épület Jellemzők</div>

      <div style="margin-bottom:12px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Alapterület:</span> <span id="hed_lbl_area" style="color:#2563eb; font-weight:700;">55 m²</span>
        </div>
        <input type="range" id="hed_area" min="30" max="120" value="55" step="1" style="width:100%; accent-color:#2563eb;" oninput="recalcHedonic()">
      </div>

      <div style="margin-bottom:12px;">
        <label style="display:block; font-size:13px; font-weight:600; color:#475569; margin-bottom:4px;">Műszaki állapot (β = +0.058 / szint):</label>
        <select id="hed_cond" style="width:100%; padding:6px 10px; border-radius:6px; border:1px solid #cbd5e1; font-size:13px; background:#fff;" onchange="recalcHedonic()">
          <option value="6">Új építésű / Újszerű (+11.6%)</option>
          <option value="5">Felújított (+5.8%)</option>
          <option value="4" selected>Jó állapotú (Bázis)</option>
          <option value="3">Közepes (-5.8%)</option>
          <option value="2">Felújítandó (-11.6%)</option>
        </select>
      </div>

      <div style="display:flex; flex-direction:column; gap:8px; margin-top:12px;">
        <label style="display:flex; align-items:center; gap:8px; font-size:13px; color:#334155; cursor:pointer;">
          <input type="checkbox" id="hed_panel" onchange="recalcHedonic()"> <b>Panel szerkezet</b> <span style="color:#dc2626; font-size:12px;">(β = -0.081, -7.8% diszkont)</span>
        </label>
        <label style="display:flex; align-items:center; gap:8px; font-size:13px; color:#334155; cursor:pointer;">
          <input type="checkbox" id="hed_lift" onchange="recalcHedonic()"> <b>Lift megléte</b> <span style="color:#059669; font-size:12px;">(β = +0.060, +6.2% prémium)</span>
        </label>
        <label style="display:flex; align-items:center; gap:8px; font-size:13px; color:#334155; cursor:pointer;">
          <input type="checkbox" id="hed_balcony" checked onchange="recalcHedonic()"> <b>Erkély kapcsolat</b> <span style="color:#059669; font-size:12px;">(β = +0.047, +4.8% prémium)</span>
        </label>
      </div>
    </div>

    <!-- 2. oszlop: Térbeli externáliák & TOD -->
    <div style="background:#f8fafc; padding:16px; border-radius:8px; border:1px solid #e2e8f0;">
      <div style="font-weight:700; color:#334155; margin-bottom:12px; font-size:14px; text-transform:uppercase; letter-spacing:0.5px;">2. Térbeli Externáliák & TOD Izokrónok</div>

      <div style="display:flex; flex-direction:column; gap:12px; margin-top:8px;">
        <label style="display:flex; align-items:flex-start; gap:8px; font-size:13px; color:#334155; cursor:pointer;">
          <input type="checkbox" id="hed_metro" checked onchange="recalcHedonic()" style="margin-top:3px;">
          <div>
            <b>Metró 10 perces sétaövezet (≤750m)</b>
            <div style="font-size:12px; color:#059669; font-weight:600;">β = +0.081 (+8.4% hálózati elérhetőségi prémium)</div>
          </div>
        </label>

        <label style="display:flex; align-items:flex-start; gap:8px; font-size:13px; color:#334155; cursor:pointer;">
          <input type="checkbox" id="hed_station" checked onchange="recalcHedonic()" style="margin-top:3px;">
          <div>
            <b>Vasútállomás 10 perces sétaövezet (≤750m)</b>
            <div style="font-size:12px; color:#2563eb; font-weight:600;">β = +0.051 (+5.2% elővárosi kapcsolat prémium)</div>
          </div>
        </label>

        <label style="display:flex; align-items:flex-start; gap:8px; font-size:13px; color:#334155; cursor:pointer;">
          <input type="checkbox" id="hed_rail_noise" onchange="recalcHedonic()" style="margin-top:3px;">
          <div>
            <b>Vasúti zaj- és rezgészóna (légvonalban ≤150m)</b>
            <div style="font-size:12px; color:#dc2626; font-weight:600;">β = -0.049 (-4.8% immissziós diszkont)</div>
          </div>
        </label>
      </div>

      <div style="background:#eff6ff; border-left:4px solid #2563eb; padding:8px 12px; border-radius:4px; font-size:12px; color:#1e40af; margin-top:16px;">
        📐 <b>Hedonikus elmélet:</b> A %-os prémium számítása: % = (exp(β) - 1) × 100, amely egzakt log-lineáris transzformáció.
      </div>
    </div>
  </div>

  <!-- Eredmény KPI-k -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:10px;">
    <div style="background:#eff6ff; border:1px solid #93c5fd; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#1e40af; text-transform:uppercase;">Összesített Hedonikus Prémium</div>
      <div id="hed_res_pct" style="font-size:26px; font-weight:800; color:#1d4ed8; margin:4px 0;">--%</div>
      <div style="font-size:11px; color:#2563eb;">A bázisárhoz viszonyítva</div>
    </div>

    <div style="background:#f0fdf4; border:1px solid #86efac; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#166534; text-transform:uppercase;">Becsült Fajlagos Ár</div>
      <div id="hed_res_nm" style="font-size:26px; font-weight:800; color:#15803d; margin:4px 0;">-- Ft/m²</div>
      <div style="font-size:11px; color:#16a34a;">Hedonikus modellérték</div>
    </div>

    <div style="background:#faf5ff; border:1px solid #d8b4fe; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#6b21a8; text-transform:uppercase;">Becsült Vételár</div>
      <div id="hed_res_tot" style="font-size:26px; font-weight:800; color:#7e22ce; margin:4px 0;">-- M Ft</div>
      <div id="hed_res_area_note" style="font-size:11px; color:#9333ea;">55 m² méretre kalkulálva</div>
    </div>

    <div style="background:#fffbeb; border:1px solid #fde68a; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#92400e; text-transform:uppercase;">Közlekedési Egyenleg</div>
      <div id="hed_res_transport" style="font-size:24px; font-weight:800; color:#b45309; margin:4px 0;">--%</div>
      <div style="font-size:11px; color:#d97706;">Metró + Vasút - Zaj netto hatás</div>
    </div>
  </div>
</div>

<script>
(function() {
  const BASE_NM = 830000; // Jó állapotú, tégla lakás bázisár Kőbányán

  function update() {
    const area = parseFloat(document.getElementById('hed_area').value);
    const condLevel = parseFloat(document.getElementById('hed_cond').value);
    const isPanel = document.getElementById('hed_panel').checked ? 1 : 0;
    const hasLift = document.getElementById('hed_lift').checked ? 1 : 0;
    const hasBalcony = document.getElementById('hed_balcony').checked ? 1 : 0;
    const isMetro = document.getElementById('hed_metro').checked ? 1 : 0;
    const isStation = document.getElementById('hed_station').checked ? 1 : 0;
    const isNoise = document.getElementById('hed_rail_noise').checked ? 1 : 0;

    document.getElementById('hed_lbl_area').innerText = area + ' m²';

    // Log-hatások összegzése
    const b_area = -0.0018 * (area - 55); // enyhe méretdiszkont
    const b_cond = 0.058 * (condLevel - 4);
    const b_panel = -0.081 * isPanel;
    const b_lift = 0.060 * hasLift;
    const b_balcony = 0.047 * hasBalcony;
    const b_metro = 0.081 * isMetro;
    const b_station = 0.051 * isStation;
    const b_noise = -0.049 * isNoise;

    const delta_log = b_area + b_cond + b_panel + b_lift + b_balcony + b_metro + b_station + b_noise;
    const pct_change = (Math.exp(delta_log) - 1.0) * 100;
    const pred_nm = BASE_NM * Math.exp(delta_log);
    const pred_tot = (pred_nm * area) / 1e6;

    const trans_log = b_metro + b_station + b_noise;
    const trans_pct = (Math.exp(trans_log) - 1.0) * 100;

    document.getElementById('hed_res_pct').innerText = (pct_change > 0 ? '+' : '') + pct_change.toFixed(1) + '%';
    document.getElementById('hed_res_nm').innerText = Math.round(pred_nm).toLocaleString('hu-HU') + ' Ft/m²';
    document.getElementById('hed_res_tot').innerText = pred_tot.toFixed(1) + ' M Ft';
    document.getElementById('hed_res_area_note').innerText = area + ' m² méretre kalkulálva';
    document.getElementById('hed_res_transport').innerText = (trans_pct > 0 ? '+' : '') + trans_pct.toFixed(1) + '%';
  }

  window.recalcHedonic = update;
  setTimeout(update, 50);
})();
</script>
'''
display(HTML(html_hedonic))